# Regularization Techniques Practice

Regularization:

- Purpose: Prevent overfitting, improve generalization
- Examples: Dropout, L1/L2 weight decay, data augmentation
- When: Mainly during training (disabled during inference)
- Effect: Add constraints or noise to reduce model complexity

In [2]:
import torch
import torch.nn as nn

# Dropout

https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

In [18]:
# ============================================================================
# PRACTICE PROBLEM 1: Dropout
# ============================================================================

class Dropout(nn.Module):
    """
    Implements Dropout regularization.
    
    During training: Randomly sets elements to zero with probability p.
                     Scales remaining elements by 1/(1-p) to maintain expected value.
    During inference: Does nothing (identity operation).
    
    Prevents co-adaptation of neurons, reduces overfitting.
    """
    
    p: float
    
    def __init__(self, p: float = 0.5) -> None:
        """
        Args:
            p: Probability of an element being zeroed. Should be in [0, 1).
        
        Returns:
            None.
        """
        super().__init__()
        assert 0 <= p < 1, "p must be in [0, 1)"
        self.p = p
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [*] (any shape).
        
        Returns:
            Output tensor. Same dimensions as input.
        
        Hint:
        1. If not training (eval mode): return x unchanged
        2. If training:
           - Create binary mask: torch.rand_like(x) > self.p
           - Mask has True where we keep values, False where we drop
           - Convert mask to float: mask.float()
           - Scale: mask / (1 - self.p)
           - Apply: x * scaled_mask
        
        Why scale by 1/(1-p)?
        - If p=0.5, we drop 50% of neurons
        - Remaining neurons must be scaled by 2x to maintain expected sum
        """
        # TODO
        if not self.training:
            return x
        
        mask = (torch.rand_like(x) > self.p).float()
        return x * mask / (1 - self.p)
        
    


# Test your implementation
def test_dropout():
    torch.manual_seed(42)  # For reproducibility
    x = torch.ones(10, 5)
    
    my_dropout = Dropout(p=0.5)
    pytorch_dropout = nn.Dropout(p=0.5)
    
    # Test training mode
    my_dropout.train()
    pytorch_dropout.train()
    
    torch.manual_seed(42)
    result_train = my_dropout(x)
    
    torch.manual_seed(42)
    expected_train = pytorch_dropout(x)
    
    print("Training mode:")
    print(f"Input (all ones): {x[0]}")
    print(f"Your result: {result_train[0]}")
    print(f"Expected:    {expected_train[0]}")
    print(f"Match: {torch.allclose(result_train, expected_train)}\n")
    
    # Test eval mode
    my_dropout.eval()
    pytorch_dropout.eval()
    
    result_eval = my_dropout(x)
    expected_eval = pytorch_dropout(x)
    
    print("Eval mode (should be unchanged):")
    print(f"Your result: {result_eval[0]}")
    print(f"Expected:    {expected_eval[0]}")
    print(f"Match: {torch.allclose(result_eval, expected_eval)}\n")



In [19]:
test_dropout()

Training mode:
Input (all ones): tensor([1., 1., 1., 1., 1.])
Your result: tensor([2., 2., 0., 2., 0.])
Expected:    tensor([0., 0., 2., 2., 2.])
Match: False

Eval mode (should be unchanged):
Your result: tensor([1., 1., 1., 1., 1.])
Expected:    tensor([1., 1., 1., 1., 1.])
Match: True



# Dropout2d (Spatial Dropout)

In [35]:
# ============================================================================
# PRACTICE PROBLEM 2: Dropout2d (Spatial Dropout)
# ============================================================================

class Dropout2d(nn.Module):
    """
    Implements 2D Dropout (Spatial Dropout).
    
    Drops entire channels instead of individual elements.
    Input shape: (B, C, H, W)
    Drops entire (H, W) feature maps.
    
    Better for convolutional layers where adjacent pixels are correlated.
    """
    
    p: float
    
    def __init__(self, p: float = 0.5) -> None:
        """
        Args:
            p: Probability of a channel being zeroed.
        
        Returns:
            None.
        """
        super().__init__()
        assert 0 <= p < 1, "p must be in [0, 1)"
        self.p = p
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Output tensor. Same dimensions as input.
        
        Hint:
        1. If eval mode: return x unchanged
        2. If training:
           - Input shape: (B, C, H, W)
           - Create mask of shape (B, C, 1, 1) - one per channel
           - Use torch.rand(B, C, 1, 1) > self.p
           - Convert to float and scale by 1/(1-p)
           - Multiply: x * mask (broadcasts over H, W)
        
        Key difference from regular Dropout:
        - Regular: drops individual elements
        - Dropout2d: drops entire channels (all H×W elements together)
        """
        # TODO
        if not self.training:
            return x
        
        B, C, H, W = x.shape
        
        mask = (torch.rand(B, C, 1, 1, device=x.device) > self.p).float()
        return x * mask / (1 - self.p)
    


# Test your implementation
def test_dropout2d():
    torch.manual_seed(42)
    x = torch.ones(2, 3, 4, 4)  # 2 images, 3 channels, 4×4
    
    my_dropout = Dropout2d(p=0.5)
    pytorch_dropout = nn.Dropout2d(p=0.5)
    
    my_dropout.train()
    pytorch_dropout.train()
    
    torch.manual_seed(42)
    result = my_dropout(x)
    
    torch.manual_seed(42)
    expected = pytorch_dropout(x)
    
    print("Training mode:")
    print(f"Input shape: {x.shape}")
    print(f"First image, first channel (should be all same - all dropped or all kept):")
    print(f"Your result:\n{result[0, 0]}")
    print(f"Expected:\n{expected[0, 0]}")
    print(f"Match: {torch.allclose(result, expected)}\n")



In [37]:
test_dropout2d()

Training mode:
Input shape: torch.Size([2, 3, 4, 4])
First image, first channel (should be all same - all dropped or all kept):
Your result:
tensor([[2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.]])
Expected:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Match: False



# AlphaDropout (for SELU activation)

In [ ]:
# ============================================================================
# PRACTICE PROBLEM 3: AlphaDropout (for SELU activation)
# ============================================================================

class AlphaDropout(nn.Module):
    """
    Implements Alpha Dropout.
    
    Special dropout that maintains mean and variance of inputs.
    Used with SELU activation to maintain self-normalizing properties.
    
    Instead of zeroing, sets dropped elements to alpha' (negative saturation value).
    """
    
    p: float
    
    def __init__(self, p: float = 0.5) -> None:
        """
        Args:
            p: Probability of an element being dropped.
        
        Returns:
            None.
        """
        super().__init__()
        assert 0 <= p < 1, "p must be in [0, 1)"
        self.p = p
        
        # Alpha and alpha_prime for SELU
        # These values maintain zero mean and unit variance
        self.alpha = -1.7580993408473766
        self.alpha_prime = -self.alpha * (1 - p)
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [*].
        
        Returns:
            Output tensor. Same dimensions as input.
        
        Hint:
        1. If eval: return x unchanged
        2. If training:
           - Create mask: torch.rand_like(x) > self.p
           - Calculate affine parameters:
             a = ((1 - p) * (1 + p * alpha_prime^2))^(-0.5)
             b = -a * alpha_prime * p
           - Apply: a * (x * mask + alpha_prime * ~mask) + b
        
        Simplified formula:
        - kept_elements = x * mask
        - dropped_elements = alpha_prime * ~mask  
        - output = a * (kept_elements + dropped_elements) + b
        """
        # TODO
        pass


# Test your implementation
def test_alpha_dropout():
    torch.manual_seed(42)
    x = torch.randn(100, 50)
    
    my_dropout = AlphaDropout(p=0.5)
    pytorch_dropout = nn.AlphaDropout(p=0.5)
    
    my_dropout.train()
    pytorch_dropout.train()
    
    torch.manual_seed(42)
    result = my_dropout(x)
    
    torch.manual_seed(42)
    expected = pytorch_dropout(x)
    
    print("Training mode:")
    print(f"Input mean: {x.mean():.4f}, std: {x.std():.4f}")
    print(f"Your result mean: {result.mean():.4f}, std: {result.std():.4f}")
    print(f"Expected mean: {expected.mean():.4f}, std: {expected.std():.4f}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

# test_alpha_dropout()

# L1 Regularization Loss

In [20]:
# ============================================================================
# PRACTICE PROBLEM 4: L1 Regularization Loss
# ============================================================================

class L1Regularization:
    """
    Implements L1 Regularization (Lasso).
    
    Adds penalty based on absolute values of weights: λ * Σ|w|
    Encourages sparsity (many weights become exactly zero).
    
    Note: This is NOT an nn.Module because it's applied to model parameters,
          not to activations.
    """
    
    lambda_: float
    
    def __init__(self, lambda_: float = 0.01) -> None:
        """
        Args:
            lambda_: Regularization strength.
        
        Returns:
            None.
        """
        self.lambda_ = lambda_
        return None
    
    def __call__(self, model: nn.Module) -> torch.Tensor:
        """
        Calculate L1 regularization loss for all model parameters.
        
        Args:
            model: PyTorch model.
        
        Returns:
            L1 regularization loss (scalar).
        
        Hint:
        1. Initialize l1_loss = 0
        2. Loop through model.parameters()
        3. For each parameter: l1_loss += torch.abs(param).sum()
        4. Return lambda_ * l1_loss
        """
        # TODO
        l1 = 0
        for param in model.parameters():
            l1 += torch.abs(param).sum()
            
        return self.lambda_ * l1
            


# Test your implementation
def test_l1_regularization():
    # Create simple model
    model = nn.Linear(10, 5)
    
    my_l1 = L1Regularization(lambda_=0.01)
    
    result = my_l1(model)
    
    # Manual calculation
    expected = 0.01 * (torch.abs(model.weight).sum() + torch.abs(model.bias).sum())
    
    print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
    print(f"Your L1 loss: {result:.4f}")
    print(f"Expected:     {expected:.4f}")
    print(f"Match: {torch.allclose(result, expected)}\n")

In [24]:
test_l1_regularization()

Model has 55 parameters
Your L1 loss: 0.0766
Expected:     0.0766
Match: True



# L2 Regularization Loss

In [25]:
# ============================================================================
# PRACTICE PROBLEM 5: L2 Regularization Loss
# ============================================================================

class L2Regularization:
    """
    Implements L2 Regularization (Ridge, Weight Decay).
    
    Adds penalty based on squared weights: λ * Σw²
    Encourages small weights (but not exactly zero).
    
    Most common regularization technique in deep learning.
    """
    
    lambda_: float
    
    def __init__(self, lambda_: float = 0.01) -> None:
        """
        Args:
            lambda_: Regularization strength.
        
        Returns:
            None.
        """
        self.lambda_ = lambda_
        return None
    
    def __call__(self, model: nn.Module) -> torch.Tensor:
        """
        Calculate L2 regularization loss for all model parameters.
        
        Args:
            model: PyTorch model.
        
        Returns:
            L2 regularization loss (scalar).
        
        Hint:
        1. Initialize l2_loss = 0
        2. Loop through model.parameters()
        3. For each parameter: l2_loss += (param ** 2).sum()
        4. Return lambda_ * l2_loss
        """
        # TODO
        l2 = 0
        
        for param in model.parameters():
            l2 += (param **2).sum()
        return self.lambda_ *  l2


# Test your implementation
def test_l2_regularization():
    model = nn.Linear(10, 5)
    
    my_l2 = L2Regularization(lambda_=0.01)
    
    result = my_l2(model)
    
    # Manual calculation
    expected = 0.01 * ((model.weight ** 2).sum() + (model.bias ** 2).sum())
    
    print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
    print(f"Your L2 loss: {result:.4f}")
    print(f"Expected:     {expected:.4f}")
    print(f"Match: {torch.allclose(result, expected)}\n")



In [138]:
test_l2_regularization()

Model has 55 parameters
Your L2 loss: 0.0181
Expected:     0.0181
Match: True



# Elastic Net Regularization (CHALLENGING!)

In [ ]:
# ============================================================================
# PRACTICE PROBLEM 6: 
# ============================================================================

class ElasticNetRegularization:
    """
    Implements Elastic Net Regularization.
    
    Combines L1 and L2: λ₁ * Σ|w| + λ₂ * Σw²
    
    Gets benefits of both:
    - L1: Feature selection (sparsity)
    - L2: Small weights, handles correlated features
    """
    
    lambda1: float
    lambda2: float
    
    def __init__(self, lambda1: float = 0.01, lambda2: float = 0.01) -> None:
        """
        Args:
            lambda1: L1 regularization strength.
            lambda2: L2 regularization strength.
        
        Returns:
            None.
        """
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        return None
    
    def __call__(self, model: nn.Module) -> torch.Tensor:
        """
        Calculate Elastic Net regularization loss.
        
        Args:
            model: PyTorch model.
        
        Returns:
            Elastic Net regularization loss (scalar).
        
        Hint:
        1. Calculate L1 loss: lambda1 * Σ|w|
        2. Calculate L2 loss: lambda2 * Σw²
        3. Return L1 + L2
        """
        # TODO
        l1_loss = 0
        l2_loss = 0
        
        for param in model.parameters():
            l1_loss += torch.abs(param).sum()
            l2_loss += (param ** 2).sum()
        return self.lambda1 * l1_loss + self.lambda2 * l2_loss
        


# Test your implementation
def test_elastic_net():
    model = nn.Linear(10, 5)
    
    my_elastic = ElasticNetRegularization(lambda1=0.01, lambda2=0.01)
    
    result = my_elastic(model)
    
    # Manual calculation
    l1 = 0.01 * (torch.abs(model.weight).sum() + torch.abs(model.bias).sum())
    l2 = 0.01 * ((model.weight ** 2).sum() + (model.bias ** 2).sum())
    expected = l1 + l2
    
    print(f"Your Elastic Net loss: {result:.4f}")
    print(f"Expected (L1 + L2):    {expected:.4f}")
    print(f"  L1 component: {l1:.4f}")
    print(f"  L2 component: {l2:.4f}")
    print(f"Match: {torch.allclose(result, expected)}\n")


In [141]:
test_elastic_net()

Your Elastic Net loss: 0.1016
Expected (L1 + L2):    0.1016
  L1 component: 0.0841
  L2 component: 0.0176
Match: True



In [ ]:
# ============================================================================
# PRACTICE PROBLEM 7: Label Smoothing (ADVANCED!)
# ============================================================================

class LabelSmoothing(nn.Module):
    """
    Implements Label Smoothing regularization.
    
    Instead of hard targets [0, 0, 1, 0], use soft targets [ε/K, ε/K, 1-ε+ε/K, ε/K]
    where K is number of classes, ε is smoothing parameter.
    
    Prevents overconfident predictions, improves generalization.
    Used in image classification (e.g., ImageNet training).
    """
    
    smoothing: float
    
    def __init__(self, smoothing: float = 0.1) -> None:
        """
        Args:
            smoothing: Label smoothing parameter (typically 0.1).
        
        Returns:
            None.
        """
        super().__init__()
        assert 0 <= smoothing < 1, "smoothing must be in [0, 1)"
        self.smoothing = smoothing
        return None
    
    def forward(self, predictions: torch.Tensor, 
                targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            predictions: Model predictions (logits, before softmax). 
                        Dimensions: [batch, num_classes].
            targets: Target class indices. Dimensions: [batch].
        
        Returns:
            Loss (scalar).
        
        Hint:
        1. Get num_classes from predictions.shape[1]
        2. Apply log_softmax to predictions
        3. Create smooth targets:
           - Start with uniform: torch.full_like(predictions, smoothing / num_classes)
           - For correct class: 1 - smoothing + smoothing / num_classes
           - Use scatter_ to set correct class values
        4. Calculate loss: -mean(smooth_targets * log_probs)
        
        Alternative simpler formula:
        - loss = (1 - smoothing) * nll_loss + smoothing * uniform_loss
        where uniform_loss = -mean(log_probs)
        """
        # TODO
        pass


# Test your implementation
def test_label_smoothing():
    torch.manual_seed(42)
    predictions = torch.randn(4, 5)  # 4 samples, 5 classes
    targets = torch.tensor([0, 2, 1, 4])
    
    my_ls = LabelSmoothing(smoothing=0.1)
    
    result = my_ls(predictions, targets)
    
    # Manual calculation (simple version)
    log_probs = torch.log_softmax(predictions, dim=1)
    nll_loss = -log_probs.gather(1, targets.unsqueeze(1)).squeeze(1).mean()
    uniform_loss = -log_probs.mean()
    expected = 0.9 * nll_loss + 0.1 * uniform_loss
    
    print(f"Your label smoothing loss: {result:.4f}")
    print(f"Expected (approximate):    {expected:.4f}")
    print(f"Close enough: {torch.allclose(result, expected, atol=1e-3)}\n")

# test_label_smoothing()


# ============================================================================
# PRACTICE PROBLEM 8: Mixup (Data Augmentation Regularization) - ADVANCED!
# ============================================================================

class Mixup:
    """
    Implements Mixup data augmentation.
    
    Creates virtual training examples by mixing pairs:
    x_mixed = λ * x_i + (1 - λ) * x_j
    y_mixed = λ * y_i + (1 - λ) * y_j
    
    where λ ~ Beta(alpha, alpha)
    
    Improves generalization, especially for small datasets.
    """
    
    alpha: float
    
    def __init__(self, alpha: float = 1.0) -> None:
        """
        Args:
            alpha: Beta distribution parameter (typically 0.2 or 1.0).
        
        Returns:
            None.
        """
        self.alpha = alpha
        return None
    
    def __call__(self, x: torch.Tensor, 
                 y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Apply Mixup to a batch.
        
        Args:
            x: Input batch. Dimensions: [batch, ...].
            y: Target labels (one-hot or indices). Dimensions: [batch, ...].
        
        Returns:
            Tuple of (mixed_x, mixed_y).
        
        Hint:
        1. Sample lambda from Beta(alpha, alpha): torch.distributions.Beta
           Or simple version: lambda ~ Uniform(0, 1) if alpha=1
        2. Create random permutation of batch: torch.randperm(batch_size)
        3. Mix inputs: x_mixed = lambda * x + (1 - lambda) * x[permutation]
        4. Mix targets: y_mixed = lambda * y + (1 - lambda) * y[permutation]
        5. Return (x_mixed, y_mixed)
        """
        # TODO
        pass


# Test your implementation
def test_mixup():
    torch.manual_seed(42)
    x = torch.randn(4, 3, 8, 8)  # 4 images
    y = torch.eye(5)[torch.tensor([0, 1, 2, 3])]  # One-hot encoded
    
    my_mixup = Mixup(alpha=1.0)
    
    torch.manual_seed(42)
    mixed_x, mixed_y = my_mixup(x, y)
    
    print(f"Original x shape: {x.shape}")
    print(f"Mixed x shape: {mixed_x.shape}")
    print(f"Original y:\n{y}")
    print(f"Mixed y:\n{mixed_y}")
    print(f"Mixed y sums to 1 per sample: {mixed_y.sum(dim=1)}\n")

# test_mixup()


# ============================================================================
# BONUS: Comparison and Usage Examples
# ============================================================================

def compare_regularizations():
    """
    Show how to use different regularization techniques together.
    """
    print("="*70)
    print("REGULARIZATION TECHNIQUES - USAGE EXAMPLE")
    print("="*70)
    print("""
# Typical training loop with multiple regularizations:

model = MyModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)  # L2 reg
l1_reg = L1Regularization(lambda_=0.001)
label_smoothing = LabelSmoothing(smoothing=0.1)

for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in dataloader:
        # Forward pass (dropout active during training)
        outputs = model(batch_x)  # Model has Dropout layers
        
        # Calculate losses
        ce_loss = criterion(outputs, batch_y)
        l1_loss = l1_reg(model)
        # Optionally: smooth_loss = label_smoothing(outputs, batch_y)
        
        # Total loss
        total_loss = ce_loss + l1_loss
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
    
    # Validation (dropout disabled)
    model.eval()
    with torch.no_grad():
        # evaluate...

# Key points:
# 1. Dropout: Automatically enabled/disabled with .train()/.eval()
# 2. L2 (weight decay): Built into optimizer
# 3. L1: Add to loss manually
# 4. Label smoothing: Use instead of regular CrossEntropyLoss
# 5. Mixup: Apply before feeding to model
    """)
    
    print("\nRegularization Comparison:")
    print("-" * 70)
    print(f"{'Technique':<20} {'Type':<15} {'When Active':<15} {'Effect'}")
    print("-" * 70)
    print(f"{'Dropout':<20} {'Stochastic':<15} {'Training only':<15} {'Prevents co-adaptation'}")
    print(f"{'L1 (Lasso)':<20} {'Parameter':<15} {'Always':<15} {'Sparsity'}")
    print(f"{'L2 (Ridge)':<20} {'Parameter':<15} {'Always':<15} {'Small weights'}")
    print(f"{'Label Smoothing':<20} {'Target':<15} {'Training only':<15} {'Soft targets'}")
    print(f"{'Mixup':<20} {'Data Aug':<15} {'Training only':<15} {'Virtual examples'}")
    print("-" * 70)

# compare_regularizations()


# ============================================================================
# BONUS: Run all tests
# ============================================================================

def run_all_tests():
    """Run all test functions"""
    print("="*70)
    print("RUNNING ALL REGULARIZATION TESTS")
    print("="*70 + "\n")
    
    tests = [
        ("Dropout", test_dropout),
        ("Dropout2d", test_dropout2d),
        ("Alpha Dropout", test_alpha_dropout),
        ("L1 Regularization", test_l1_regularization),
        ("L2 Regularization", test_l2_regularization),
        ("Elastic Net", test_elastic_net),
        ("Label Smoothing", test_label_smoothing),
        ("Mixup", test_mixup),
    ]
    
    for name, test_func in tests:
        print(f"--- {name} ---")
        try:
            test_func()
        except Exception as e:
            print(f"❌ Error: {e}\n")
    
    print("="*70)
    print("TESTS COMPLETE")
    print("="*70)

# Uncomment to run all tests:
# run_all_tests()
# compare_regularizations()


# ============================================================================
# SOLUTIONS (Don't look until you've tried!)
# ============================================================================

# Scroll down for solutions...
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#

"""
SOLUTIONS:

# 1. Dropout
def forward(self, x):
    if not self.training:
        return x
    
    mask = (torch.rand_like(x) > self.p).float()
    return x * mask / (1 - self.p)


# 2. Dropout2d
def forward(self, x):
    if not self.training:
        return x
    
    B, C, H, W = x.shape
    mask = (torch.rand(B, C, 1, 1, device=x.device) > self.p).float()
    return x * mask / (1 - self.p)


# 3. AlphaDropout
def forward(self, x):
    if not self.training:
        return x
    
    mask = torch.rand_like(x) > self.p
    
    # Calculate affine parameters
    a = ((1 - self.p) * (1 + self.p * self.alpha_prime ** 2)) ** (-0.5)
    b = -a * self.alpha_prime * self.p
    
    # Apply dropout
    return a * (x * mask + self.alpha_prime * ~mask) + b


# 4. L1 Regularization
def __call__(self, model):
    l1_loss = 0
    for param in model.parameters():
        l1_loss += torch.abs(param).sum()
    return self.lambda_ * l1_loss


# 5. L2 Regularization
def __call__(self, model):
    l2_loss = 0
    for param in model.parameters():
        l2_loss += (param ** 2).sum()
    return self.lambda_ * l2_loss


# 6. Elastic Net
def __call__(self, model):
    l1_loss = 0
    l2_loss = 0
    for param in model.parameters():
        l1_loss += torch.abs(param).sum()
        l2_loss += (param ** 2).sum()
    return self.lambda1 * l1_loss + self.lambda2 * l2_loss


# 7. Label Smoothing
def forward(self, predictions, targets):
    num_classes = predictions.shape[1]
    log_probs = torch.log_softmax(predictions, dim=1)
    
    # Simple formula
    nll_loss = -log_probs.gather(1, targets.unsqueeze(1)).squeeze(1).mean()
    uniform_loss = -log_probs.mean()
    
    return (1 - self.smoothing) * nll_loss + self.smoothing * uniform_loss


# 7. Label Smoothing (Alternative - full implementation)
def forward(self, predictions, targets):
    batch_size, num_classes = predictions.shape
    
    # Apply log softmax
    log_probs = torch.log_softmax(predictions, dim=1)
    
    # Create smooth targets
    smooth_targets = torch.full_like(log_probs, self.smoothing / num_classes)
    smooth_targets.scatter_(1, targets.unsqueeze(1), 
                           1.0 - self.smoothing + self.smoothing / num_classes)
    
    # Calculate loss
    loss = -(smooth_targets * log_probs).sum(dim=1).mean()
    return loss


# 8. Mixup
def __call__(self, x, y):
    batch_size = x.shape[0]
    
    # Sample lambda
    if self.alpha > 0:
        lam = torch.from_numpy(
            np.random.beta(self.alpha, self.alpha, size=1)
        ).float().item()
    else:
        lam = 1.0
    
    # Random permutation
    index = torch.randperm(batch_size)
    
    # Mix inputs and targets
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]
    
    return mixed_x, mixed_y


# 8. Mixup (Simpler version without numpy)
def __call__(self, x, y):
    batch_size = x.shape[0]
    
    # Simple uniform lambda for alpha=1
    lam = torch.rand(1).item()
    
    # Random permutation
    index = torch.randperm(batch_size)
    
    # Mix
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]
    
    return mixed_x, mixed_y
"""